## Etapa 3 - Data Quality

### Contexto inicial del dataset

In [2]:
import pandas as pd
import numpy as np

 1) bank-full.csv 45211 registrosy 17 variables de entrada, ordenado por fecha (versión anterior de este conjunto de datos con menos variables de entrada).

In [3]:
df = pd.read_csv(r"C:\Users\USUARIO\Downloads\bank-full.csv", sep=";")

In [4]:
# Ver los datos
df.sample(10)

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
19598,40,technician,married,secondary,no,998,no,no,cellular,7,aug,133,7,-1,0,unknown,no
14537,60,retired,married,secondary,no,0,no,no,cellular,15,jul,94,3,-1,0,unknown,no
9043,48,retired,single,secondary,no,3458,no,no,unknown,5,jun,292,1,-1,0,unknown,no
32194,51,unemployed,married,tertiary,no,1801,no,no,cellular,16,apr,838,3,-1,0,unknown,yes
34594,36,blue-collar,married,secondary,no,108,yes,no,cellular,5,may,348,1,-1,0,unknown,no
6756,28,blue-collar,single,secondary,no,6307,yes,no,unknown,28,may,615,1,-1,0,unknown,no
12325,57,self-employed,married,secondary,no,35,no,yes,unknown,26,jun,215,3,-1,0,unknown,no
26046,52,technician,married,unknown,no,9687,yes,no,cellular,19,nov,222,3,174,1,failure,no
6081,36,blue-collar,married,secondary,no,3935,yes,yes,unknown,27,may,220,6,-1,0,unknown,no
15466,33,services,married,secondary,no,-27,yes,no,cellular,18,jul,216,2,-1,0,unknown,no


In [5]:
# Dimensionalidad
print(f"Dimensionalidad\nFilas: {df.shape[0]}\nColumnas: {df.shape[1]}")

Dimensionalidad
Filas: 45211
Columnas: 17


In [6]:
# Tipos de datos
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   age        45211 non-null  int64
 1   job        45211 non-null  str  
 2   marital    45211 non-null  str  
 3   education  45211 non-null  str  
 4   default    45211 non-null  str  
 5   balance    45211 non-null  int64
 6   housing    45211 non-null  str  
 7   loan       45211 non-null  str  
 8   contact    45211 non-null  str  
 9   day        45211 non-null  int64
 10  month      45211 non-null  str  
 11  duration   45211 non-null  int64
 12  campaign   45211 non-null  int64
 13  pdays      45211 non-null  int64
 14  previous   45211 non-null  int64
 15  poutcome   45211 non-null  str  
 16  y          45211 non-null  str  
dtypes: int64(7), str(10)
memory usage: 5.9 MB


* 7 variables numéricas
* 10 categóricas
* ninguna presenta NaN.

job	        B
marital	    B
education	B
default	    B  ??
balance	    B
housing	    B Binario
loan	    B Binario
contact	    B
day	        B
month	    B 1-12
duration	B ??
campaign	B ??
pdays	    B
previous	B
poutcome	B ??
y           B Binario

### 3.1 Valores Faltantes

In [7]:
# Valores nulos
df.isna().sum()

age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
y            0
dtype: int64

No se identificaron valores faltantes representados mediante NaN

### 3.2 Valores faltantes codificados

In [8]:
# Valores faltantes codificados como "unknown"

for col in df.select_dtypes(include="object").columns:
    unknown_count = (df[col] == "unknown").sum()
    
    if unknown_count > 0:
        print(
            f"{col}: {unknown_count} registros "
            f"({unknown_count / len(df) * 100:.2f}%)"
        )

job: 288 registros (0.64%)
education: 1857 registros (4.11%)
contact: 13020 registros (28.80%)
poutcome: 36959 registros (81.75%)


C:\Users\USUARIO\AppData\Local\Temp\ipykernel_26836\1985257398.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:


Se identificaron valores faltantes que no se encuentran representados como NaN, sino mediante la categoría "unknown". Este comportamiento es especialmente relevante en variables categóricas, por lo que se realizó un conteo por variable para determinar su impacto.

Se encontró que job presenta únicamente un 0.64% de valores "unknown", mientras que education presenta un 4.11%. En ambos casos, la proporción es relativamente baja.

La variable contact presenta un 28.80% de registros desconocidos. En este caso, el valor "unknown" puede representar que el tipo de contacto utilizado no fue registrado, por lo que no debe eliminarse automáticamente.

El caso más significativo corresponde a poutcome, donde el 81.75% de los registros tiene el valor "unknown". Este porcentaje elevado no necesariamente representa un error de calidad, ya que esta variable corresponde al resultado de una campaña de marketing anterior y "unknown" puede indicar que no existe información sobre una campaña previa.

Por esta razón, se decidió mantener inicialmente la categoría "unknown" como una categoría válida, evitando eliminar registros y evitando imputaciones arbitrarias. Posteriormente, durante el Feature Engineering, se evaluará si esta categoría aporta información predictiva al modelo.

### 3.3 Duplicados

In [9]:
# Registros duplicados

duplicados = df.duplicated().sum()

print(f"Registros duplicados: {duplicados}")
print(f"Porcentaje de duplicados: {(duplicados / len(df)) * 100:.2f}%")

Registros duplicados: 0
Porcentaje de duplicados: 0.00%


No se encontraron registros duplicados.

### 3.4 Registros inconsistentes

In [21]:
# Revisión de combinaciones entre variables binarias

variables_binarias = ["default", "housing", "loan", "y"]

for col in variables_binarias:
    print(f"\n{col}:")
    print(df[col].value_counts())


default:
default
no     44396
yes      815
Name: count, dtype: int64

housing:
housing
yes    25130
no     20081
Name: count, dtype: int64

loan:
loan
no     37967
yes     7244
Name: count, dtype: int64

y:
y
no     39922
yes     5289
Name: count, dtype: int64


In [22]:
# Posibles inconsistencias entre previous y pdays

inconsistencia_1 = df[
    (df["previous"] == 0) &
    (df["pdays"] != -1)
]

inconsistencia_2 = df[
    (df["previous"] > 0) &
    (df["pdays"] == -1)
]

print("previous = 0 y pdays != -1:", len(inconsistencia_1))
print("previous > 0 y pdays = -1:", len(inconsistencia_2))

previous = 0 y pdays != -1: 0
previous > 0 y pdays = -1: 0


Se revisaron posibles inconsistencias entre variables relacionadas con el historial de contacto del cliente. En particular, se verificó la relación entre previous y pdays, debido a que ambas variables describen información relacionada con contactos anteriores.

No se encontraron registros donde previous = 0 y pdays tuviera un valor diferente de -1, ni registros donde previous > 0 y pdays = -1. Por lo tanto, no se identificaron inconsistencias en esta relación.

También se verificaron las variables binarias default, housing, loan y y, encontrándose únicamente las categorías esperadas (yes y no).

### 3.5 Tipos incorrectos

In [23]:
# Tipos de datos de las variables

print(df.dtypes)

age          int64
job            str
marital        str
education      str
default        str
balance      int64
housing        str
loan           str
contact        str
day          int64
month          str
duration     int64
campaign     int64
pdays        int64
previous     int64
poutcome       str
y              str
dtype: object


In [24]:
# Tipos de datos esperados

tipos_esperados = {
    "age": "numérico",
    "job": "categórico",
    "marital": "categórico",
    "education": "categórico",
    "default": "categórico",
    "balance": "numérico",
    "housing": "categórico",
    "loan": "categórico",
    "contact": "categórico",
    "day": "numérico",
    "month": "categórico",
    "duration": "numérico",
    "campaign": "numérico",
    "pdays": "numérico",
    "previous": "numérico",
    "poutcome": "categórico",
    "y": "categórico"
}

for col, tipo in tipos_esperados.items():
    tipo_real = "numérico" if pd.api.types.is_numeric_dtype(df[col]) else "categórico"
    estado = "OK" if tipo_real == tipo else "REVISAR"

    print(f"{col}: esperado={tipo} | encontrado={tipo_real} | {estado}")

age: esperado=numérico | encontrado=numérico | OK
job: esperado=categórico | encontrado=categórico | OK
marital: esperado=categórico | encontrado=categórico | OK
education: esperado=categórico | encontrado=categórico | OK
default: esperado=categórico | encontrado=categórico | OK
balance: esperado=numérico | encontrado=numérico | OK
housing: esperado=categórico | encontrado=categórico | OK
loan: esperado=categórico | encontrado=categórico | OK
contact: esperado=categórico | encontrado=categórico | OK
day: esperado=numérico | encontrado=numérico | OK
month: esperado=categórico | encontrado=categórico | OK
duration: esperado=numérico | encontrado=numérico | OK
campaign: esperado=numérico | encontrado=numérico | OK
pdays: esperado=numérico | encontrado=numérico | OK
previous: esperado=numérico | encontrado=numérico | OK
poutcome: esperado=categórico | encontrado=categórico | OK
y: esperado=categórico | encontrado=categórico | OK


La revisión de los tipos de datos permitió comprobar que las variables numéricas se encuentran almacenadas como tipos numéricos y que las variables categóricas se encuentran almacenadas como variables de texto (`object`).

Las 17 variables evaluadas coinciden con el tipo de dato esperado de acuerdo con su naturaleza. No se identificaron variables que requieran conversión de tipo en esta etapa.

### 3.6 Categorías inconsistentes

In [25]:
# Revisión de espacios y diferencias de escritura

columnas_categoricas = df.select_dtypes(include="object").columns

for col in columnas_categoricas:
    valores = df[col].dropna().astype(str)

    con_espacios = valores[
        valores.str.strip() != valores
    ]

    con_mayusculas = valores[
        valores != valores.str.lower()
    ]

    print(f"\n--- {col} ---")
    print(f"Valores con espacios al inicio/final: {len(con_espacios)}")
    print(f"Valores con mayúsculas: {len(con_mayusculas)}")

C:\Users\USUARIO\AppData\Local\Temp\ipykernel_26836\304826677.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  columnas_categoricas = df.select_dtypes(include="object").columns



--- job ---
Valores con espacios al inicio/final: 0
Valores con mayúsculas: 0

--- marital ---
Valores con espacios al inicio/final: 0
Valores con mayúsculas: 0

--- education ---
Valores con espacios al inicio/final: 0
Valores con mayúsculas: 0

--- default ---
Valores con espacios al inicio/final: 0
Valores con mayúsculas: 0

--- housing ---
Valores con espacios al inicio/final: 0
Valores con mayúsculas: 0

--- loan ---
Valores con espacios al inicio/final: 0
Valores con mayúsculas: 0

--- contact ---
Valores con espacios al inicio/final: 0
Valores con mayúsculas: 0

--- month ---
Valores con espacios al inicio/final: 0
Valores con mayúsculas: 0

--- poutcome ---
Valores con espacios al inicio/final: 0
Valores con mayúsculas: 0

--- y ---
Valores con espacios al inicio/final: 0
Valores con mayúsculas: 0


In [26]:
# Categorías encontradas en las variables categóricas

for col in columnas_categoricas:
    print(f"\n--- {col} ---")
    print(sorted(df[col].dropna().unique()))


--- job ---
['admin.', 'blue-collar', 'entrepreneur', 'housemaid', 'management', 'retired', 'self-employed', 'services', 'student', 'technician', 'unemployed', 'unknown']

--- marital ---
['divorced', 'married', 'single']

--- education ---
['primary', 'secondary', 'tertiary', 'unknown']

--- default ---
['no', 'yes']

--- housing ---
['no', 'yes']

--- loan ---
['no', 'yes']

--- contact ---
['cellular', 'telephone', 'unknown']

--- month ---
['apr', 'aug', 'dec', 'feb', 'jan', 'jul', 'jun', 'mar', 'may', 'nov', 'oct', 'sep']

--- poutcome ---
['failure', 'other', 'success', 'unknown']

--- y ---
['no', 'yes']


La revisión de las variables categóricas no identificó diferencias en la escritura de las categorías, espacios innecesarios ni variaciones entre mayúsculas y minúsculas. Todas las variables presentaron categorías.

También se verificó que las categorías observadas corresponden a los valores esperados para cada variable. Los valores `unknown` presentes en `job`, `education`, `contact` y `poutcome` no se consideran inconsistencias categóricas, ya que fueron identificados previamente como valores utilizados para representar información desconocida.

### 3.7 Fechas Inválidas

In [27]:
# Validación del día

dias_invalidos = df[
    (df["day"] < 1) |
    (df["day"] > 31)
]

print("Días inválidos:", len(dias_invalidos))

Días inválidos: 0


In [28]:
# Validación de meses

meses_validos = [
    "jan", "feb", "mar", "apr",
    "may", "jun", "jul", "aug",
    "sep", "oct", "nov", "dec"
]

meses_invalidos = df[
    ~df["month"].isin(meses_validos)
]

print("Meses inválidos:", len(meses_invalidos))

Meses inválidos: 0


In [29]:
# Validación de combinaciones mes-día

dias_por_mes = {
    "jan": 31,
    "feb": 28,
    "mar": 31,
    "apr": 30,
    "may": 31,
    "jun": 30,
    "jul": 31,
    "aug": 31,
    "sep": 30,
    "oct": 31,
    "nov": 30,
    "dec": 31
}

fechas_invalidas = df[
    df.apply(lambda x: x["day"] > dias_por_mes[x["month"]], axis=1)
]

print("Registros con combinaciones mes-día inválidas:", len(fechas_invalidas))

Registros con combinaciones mes-día inválidas: 0


El dataset no contiene una variable de fecha completa, sino las variables `day` y `month`, por lo que la validación se realizó de acuerdo con los dominios disponibles.

La variable `day` no presentó valores fuera del rango válido de 1 a 31, con 0 registros inválidos. La variable `month` tampoco presentó categorías fuera de los 12 meses esperados, con 0 registros inválidos.

Debido a que el dataset no contiene una fecha completa, no es posible realizar validaciones adicionales relacionadas con fechas específicas.

### 3.8 Datos imposibles

In [14]:
# Validación de rangos esperados

print("Registros con posibles valores fuera de rango:\n")

print("Edad fuera de rango:", ((df["age"] < 18) | (df["age"] > 100)).sum())
print("Día fuera de rango:", ((df["day"] < 1) | (df["day"] > 31)).sum())
print("Duración negativa:", (df["duration"] < 0).sum())
print("Campaign menor que 1:", (df["campaign"] < 1).sum())
print("Previous negativo:", (df["previous"] < 0).sum())

Registros con posibles valores fuera de rango:

Edad fuera de rango: 0
Día fuera de rango: 0
Duración negativa: 0
Campaign menor que 1: 0
Previous negativo: 0


No se identificaron registros con datos imposibles en las variables numéricas analizadas. Las edades se encuentran dentro del rango esperado, los días son válidos, la duración no presenta valores negativos y las variables campaign y previous respetan sus restricciones mínimas.

In [15]:
# Revisión del valor especial -1 en pdays

pdays_unknown = (df["pdays"] == -1).sum()
pdays_porcentaje = (pdays_unknown / len(df)) * 100

print(f"Registros con pdays = -1: {pdays_unknown}")
print(f"Porcentaje del dataset: {pdays_porcentaje:.2f}%")

Registros con pdays = -1: 36954
Porcentaje del dataset: 81.74%


Se identificaron 36,954 registros con un valor de -1, equivalente al 81.74% del dataset. Este valor no representa una cantidad negativa de días, sino que corresponde a una codificación utilizada por el dataset para indicar que el cliente no había sido contactado previamente.

Por lo tanto, se considera un valor válido dentro del contexto de la variable y no requiere imputación, eliminación o transformación durante esta etapa de Data Quality.

### 3.9 Valores Extremos (Outliers)



In [18]:
# Detección de valores extremos mediante IQR

variables_outliers = [
    "age",
    "balance",
    "duration",
    "campaign",
    "previous"
]

for col in variables_outliers:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR

    outliers = (
        (df[col] < limite_inferior) |
        (df[col] > limite_superior)
    ).sum()

    porcentaje = (outliers / len(df)) * 100

    print(f"\n--- {col} ---")
    print(f"Q1: {Q1:.2f}")
    print(f"Q3: {Q3:.2f}")
    print(f"IQR: {IQR:.2f}")
    print(f"Límite inferior: {limite_inferior:.2f}")
    print(f"Límite superior: {limite_superior:.2f}")
    print(f"Cantidad de posibles outliers: {outliers}")
    print(f"Porcentaje: {porcentaje:.2f}%")


--- age ---
Q1: 33.00
Q3: 48.00
IQR: 15.00
Límite inferior: 10.50
Límite superior: 70.50
Cantidad de posibles outliers: 487
Porcentaje: 1.08%

--- balance ---
Q1: 72.00
Q3: 1428.00
IQR: 1356.00
Límite inferior: -1962.00
Límite superior: 3462.00
Cantidad de posibles outliers: 4729
Porcentaje: 10.46%

--- duration ---
Q1: 103.00
Q3: 319.00
IQR: 216.00
Límite inferior: -221.00
Límite superior: 643.00
Cantidad de posibles outliers: 3235
Porcentaje: 7.16%

--- campaign ---
Q1: 1.00
Q3: 3.00
IQR: 2.00
Límite inferior: -2.00
Límite superior: 6.00
Cantidad de posibles outliers: 3064
Porcentaje: 6.78%

--- previous ---
Q1: 0.00
Q3: 0.00
IQR: 0.00
Límite inferior: 0.00
Límite superior: 0.00
Cantidad de posibles outliers: 8257
Porcentaje: 18.26%


La aplicación del método del rango intercuartílico (IQR) permitió identificar posibles valores extremos en algunas variables numéricas. En age se identificaron 487 registros (1.08%), mientras que balance, duration y campaign presentaron 10.46%, 7.16% y 6.78% de posibles valores extremos, respectivamente.

La identificación de estos valores no implica que sean errores en los datos. Por ejemplo, los valores elevados de balance pueden corresponder a clientes con saldos altos, mientras que los valores elevados de duration y campaign pueden representar situaciones reales relacionadas con la interacción comercial. Por esta razón, estos registros no serán eliminados automáticamente.

En el caso de previous, el primer y tercer cuartil son iguales a cero, por lo que el IQR también es cero. Como consecuencia, el método identifica como posibles outliers todos los valores mayores que cero. Este resultado está relacionado con la distribución de la variable y no constituye evidencia suficiente para considerar dichos registros como errores.

En conclusión, los valores identificados mediante IQR serán conservados en esta etapa, ya que no se cuenta con evidencia suficiente para determinar que correspondan a registros incorrectos. Su comportamiento será considerado posteriormente durante el análisis de distribución y skewness.

### 3.10 Cardinalidad 

In [ ]:
# Cardinalidad de las variables
for col in df.columns:
    print(f"{col}: {len(df[col].unique())}")


age: 77
job: 12
marital: 3
education: 4
default: 2


balance: 7168
housing: 2
loan: 2
contact: 3
day: 31
month: 12
duration: 1573
campaign: 48
pdays: 559
previous: 41
poutcome: 4
y: 2


In [11]:
cols = df[["job", "marital", "education", "default", "housing", "loan", "contact"]]

for col in cols:
    print((cols[col].value_counts() * 100) / len(cols[col]), "\n")


job
blue-collar      21.525735
management       20.919688
technician       16.803433
admin.           11.437482
services          9.188029
retired           5.007631
self-employed     3.492513
entrepreneur      3.289023
unemployed        2.882042
housemaid         2.742695
student           2.074716
unknown           0.637013
Name: count, dtype: float64 

marital
married     60.193316
single      28.289576
divorced    11.517109
Name: count, dtype: float64 

education
secondary    51.319369
tertiary     29.419831
primary      15.153392
unknown       4.107407
Name: count, dtype: float64 

default
no     98.197341
yes     1.802659
Name: count, dtype: float64 

housing
yes    55.583818
no     44.416182
Name: count, dtype: float64 

loan
no     83.977351
yes    16.022649
Name: count, dtype: float64 

contact
cellular     64.774059
unknown      28.798301
telephone     6.427639
Name: count, dtype: float64 



La cardinalidad permitió identificar la cantidad de valores únicos presentes en cada variable. Las variables categóricas presentan una cardinalidad baja, entre 2 y 12 categorías, mientras que las variables numéricas presentan una mayor cantidad de valores únicos, especialmente balance, duration y pdays. No se identificaron cardinalidades inesperadamente elevadas en las variables categóricas que indiquen problemas evidentes de calidad. La variable objetivo y presenta una cardinalidad de 2, correspondiente a las categorías yes y no.

### 3.11 Skewness (Asimetría)

In [19]:
# Cálculo de skewness para las variables numéricas

variables_numericas = df.select_dtypes(include="number").columns

skewness = df[variables_numericas].skew().sort_values(ascending=False)

print("Skewness de las variables numéricas:")
print(skewness.round(2))

Skewness de las variables numéricas:
previous    41.85
balance      8.36
campaign     4.90
duration     3.14
pdays        2.62
age          0.68
day          0.09
dtype: float64


El análisis de skewness muestra que varias de las variables numéricas presentan distribuciones con una asimetría considerable. Las variables previous, balance, campaign, duration y pdays presentan valores positivos de skewness superiores a 1, lo que indica una distribución sesgada hacia la derecha.

previous presenta el mayor nivel de asimetría, con un valor de 41.85. Este resultado está relacionado con la alta concentración de registros con valor 0 observada anteriormente, por lo que no se considera por sí mismo evidencia de un problema de calidad.

balance, campaign, duration y pdays también presentan una asimetría elevada. Estos resultados son consistentes con la presencia de valores extremos identificados mediante el método IQR. Sin embargo, los valores extremos pueden representar situaciones reales dentro del contexto del negocio, por lo que no se eliminarán automáticamente.

La variable age presenta una asimetría moderada (0.68), mientras que day presenta una distribución cercana a simétrica, con un valor de 0.09.

En general, la presencia de asimetría se registra como una característica de las distribuciones del conjunto de datos y será considerada posteriormente al momento de seleccionar las técnicas de transformación y modelado.

### 3.12 Errores de unidad

In [30]:
# Revisión de las unidades de las variables numéricas

print("Unidades esperadas:")
print("age      -> años")
print("balance  -> euros")
print("day      -> día del mes")
print("duration -> segundos")
print("campaign -> cantidad de contactos")
print("pdays    -> días")
print("previous -> cantidad de contactos anteriores")

Unidades esperadas:
age      -> años
balance  -> euros
day      -> día del mes
duration -> segundos
campaign -> cantidad de contactos
pdays    -> días
previous -> cantidad de contactos anteriores


In [31]:
# Rangos observados de las variables

print("Rangos observados:\n")

print(f"age: {df['age'].min()} - {df['age'].max()} años")
print(f"balance: {df['balance'].min()} - {df['balance'].max()} euros")
print(f"day: {df['day'].min()} - {df['day'].max()}")
print(f"duration: {df['duration'].min()} - {df['duration'].max()} segundos")
print(f"campaign: {df['campaign'].min()} - {df['campaign'].max()} contactos")
print(f"pdays: {df['pdays'].min()} - {df['pdays'].max()} días")
print(f"previous: {df['previous'].min()} - {df['previous'].max()} contactos")

Rangos observados:

age: 18 - 95 años
balance: -8019 - 102127 euros
day: 1 - 31
duration: 0 - 4918 segundos
campaign: 1 - 63 contactos
pdays: -1 - 871 días
previous: 0 - 275 contactos


La revisión de las unidades de las variables numéricas no evidenció errores de unidad ni mezcla de escalas dentro del dataset. Los rangos observados son compatibles con las unidades correspondientes: `age` está expresada en años, `balance` en euros, `day` como día del mes, `duration` en segundos, `campaign` y `previous` como cantidades de contactos, y `pdays` en días.

El valor negativo observado en `balance` no se considera un error de unidad, ya que corresponde a un saldo negativo. De igual forma, `pdays = -1` corresponde a un valor especial de la variable y fue analizado previamente.


### 3.13 Leakage

In [32]:
# Variables disponibles para evaluar posibles casos de leakage

print("Variable objetivo:", "y")
print("\nVariables predictoras:")

for col in df.columns:
    if col != "y":
        print("-", col)

Variable objetivo: y

Variables predictoras:
- age
- job
- marital
- education
- default
- balance
- housing
- loan
- contact
- day
- month
- duration
- campaign
- pdays
- previous
- poutcome


In [34]:
# Revisión de posibles variables con leakage

variables_revision = [
    "duration",
    "campaign",
    "pdays",
    "previous",
    "poutcome"
]

for col in variables_revision:
    print(f"\n--- {col} ---")
    
    if pd.api.types.is_numeric_dtype(df[col]):
        print(df.groupby("y")[col].mean())
    else:
        print(pd.crosstab(df[col], df["y"], normalize="index") * 100)


--- duration ---
y
no     221.182806
yes    537.294574
Name: duration, dtype: float64

--- campaign ---
y
no     2.846350
yes    2.141047
Name: campaign, dtype: float64

--- pdays ---
y
no     36.421372
yes    68.702968
Name: pdays, dtype: float64

--- previous ---
y
no     0.502154
yes    1.170354
Name: previous, dtype: float64

--- poutcome ---
y                no        yes
poutcome                      
failure   87.390329  12.609671
other     83.315217  16.684783
success   35.274653  64.725347
unknown   90.838497   9.161503


Durante la revisión de leakage se analizaron las variables `duration`, `campaign`, `pdays`, `previous` y `poutcome`.

No se identificó leakage evidente en `campaign`, `pdays`, `previous` ni `poutcome`, ya que corresponden a información relacionada con la campaña actual o con contactos anteriores.

La variable `duration` fue identificada como una posible fuente de leakage debido a que su valor solamente se conoce después de finalizar el contacto con el cliente. Por lo tanto, si el modelo se utiliza para predecir la probabilidad de contratación antes de realizar la llamada, esta variable no debería utilizarse como predictor.

### 3.14 Desbalance (Imbalance)

In [36]:
# Distribución de la variable objetivo

conteo_y = df["y"].value_counts()
porcentaje_y = df["y"].value_counts(normalize=True) * 100

print("Cantidad de registros por clase:\n")
print(conteo_y)

print("\nPorcentaje por clase:\n")
print(porcentaje_y.round(2))

Cantidad de registros por clase:

y
no     39922
yes     5289
Name: count, dtype: int64

Porcentaje por clase:

y
no     88.3
yes    11.7
Name: proportion, dtype: float64


La variable objetivo `y` presenta un desbalance entre sus dos clases. La clase `no` representa aproximadamente el 88.30% de los registros, mientras que la clase `yes` representa aproximadamente el 11.70%.

Por lo tanto, existe un desbalance considerable en la variable objetivo, con `yes` como clase minoritaria. Esta situación deberá considerarse durante la construcción y evaluación de los modelos, ya que utilizar únicamente la exactitud (accuracy) podría proporcionar una visión poco representativa del desempeño del modelo.

### 3.15 Correlación Excesiva

In [37]:
# Selección de variables numéricas

variables_numericas = df.select_dtypes(include="number").columns

correlacion = df[variables_numericas].corr()

print(correlacion.round(2))

           age  balance   day  duration  campaign  pdays  previous
age       1.00     0.10 -0.01     -0.00      0.00  -0.02      0.00
balance   0.10     1.00  0.00      0.02     -0.01   0.00      0.02
day      -0.01     0.00  1.00     -0.03      0.16  -0.09     -0.05
duration -0.00     0.02 -0.03      1.00     -0.08  -0.00      0.00
campaign  0.00    -0.01  0.16     -0.08      1.00  -0.09     -0.03
pdays    -0.02     0.00 -0.09     -0.00     -0.09   1.00      0.45
previous  0.00     0.02 -0.05      0.00     -0.03   0.45      1.00


La matriz de correlación muestra que no existen pares de variables numéricas con una correlación absoluta igual o superior a 0.80, criterio utilizado para identificar posibles casos de correlación excesiva.

La mayor correlación observada corresponde a `pdays` y `previous`, con un valor de 0.45. Aunque estas variables presentan una relación positiva moderada, no se considera suficientemente alta como para indicar una redundancia excesiva.

Por lo tanto, no se identifican problemas de correlación excesiva entre las variables numéricas analizadas.

### 3.17 Anomalías estadísticas

In [39]:
# Selección de variables numéricas para detectar anomalías

variables_numericas = [
    "age",
    "balance",
    "day",
    "duration",
    "campaign",
    "pdays",
    "previous"
]

df_numericas = df[variables_numericas].copy()

In [40]:
# Cálculo del Z-score

from scipy.stats import zscore

z_scores = df_numericas.apply(zscore)

print(z_scores.head())

        age   balance       day  duration  campaign     pdays  previous
0  1.606965  0.256419 -1.298476  0.011016 -0.569351 -0.411453  -0.25194
1  0.288529 -0.437895 -1.298476 -0.416127 -0.569351 -0.411453  -0.25194
2 -0.747384 -0.446762 -1.298476 -0.707361 -0.569351 -0.411453  -0.25194
3  0.571051  0.047205 -1.298476 -0.645231 -0.569351 -0.411453  -0.25194
4 -0.747384 -0.447091 -1.298476 -0.233620 -0.569351 -0.411453  -0.25194


In [41]:
# Cantidad de variables con Z-score superior a 3 en cada registro

cantidad_anomalias = (z_scores.abs() > 3).sum(axis=1)

print("Registros con al menos una anomalía estadística:",
      (cantidad_anomalias > 0).sum())

print("Registros con dos o más anomalías estadísticas:",
      (cantidad_anomalias >= 2).sum())

print("Registros con tres o más anomalías estadísticas:",
      (cantidad_anomalias >= 3).sum())

Registros con al menos una anomalía estadística: 5002
Registros con dos o más anomalías estadísticas: 228
Registros con tres o más anomalías estadísticas: 4


In [42]:
# Cantidad de posibles anomalías por variable

anomalias_por_variable = (z_scores.abs() > 3).sum()

print("Posibles anomalías por variable:\n")
print(anomalias_por_variable)

Posibles anomalías por variable:

age          381
balance      745
day            0
duration     963
campaign     840
pdays       1723
previous     582
dtype: int64


Utilizando un criterio de |Z-score| > 3, se identificaron 5,002 registros con al menos una variable estadísticamente inusual. Sin embargo, esto no implica que dichos registros sean incorrectos, ya que varias variables del dataset presentan distribuciones sesgadas.

Al analizar simultáneamente las variables numéricas, se encontraron 228 registros con anomalías en dos o más variables y solamente 4 registros con anomalías en tres o más variables.

La mayor cantidad de posibles anomalías corresponde a `pdays`, con 1,723 registros, seguida de `duration`, `campaign` y `balance`. Estos resultados deben interpretarse con precaución debido a la distribución de las variables y no constituyen por sí mismos evidencia de errores en los datos.